# Evaluatie Functie 1: Text-to-SQL — Execution Accuracy

**Methode:** de LLM genereert een SQL-query op basis van een natuurlijke taalvraag. Die query wordt uitgevoerd op de database en het resultaat wordt vergeleken met het resultaat van een handmatig geschreven gouden SQL-query. Als beide resultaten gelijk zijn → CORRECT.



## 1. Imports & configuratie

In [50]:
import importlib
import system_prompt
importlib.reload(system_prompt)
from system_prompt import SYSTEM_PROMPT

In [51]:
import json
import sqlite3
import sys
from pathlib import Path

from dotenv import load_dotenv

# ── Paden: notebook heeft geen __file__, dus we detecteren de projectmap via cwd ──
_cwd = Path.cwd()
PROJECT_DIR  = _cwd.parent if _cwd.name == "evaluatie" else _cwd
EVAL_DIR     = PROJECT_DIR / "evaluatie"
DB_PATH      = str(PROJECT_DIR / "data" / "agent.db")
DATASET_PATH = EVAL_DIR / "golden_dataset_f1.json"

print(f"PROJECT_DIR  : {PROJECT_DIR}")
print(f"DB_PATH      : {DB_PATH}")
print(f"DATASET_PATH : {DATASET_PATH}")

load_dotenv(PROJECT_DIR / ".env")

sys.path.insert(0, str(PROJECT_DIR / "src"))
from system_prompt import SYSTEM_PROMPT

from anthropic import Anthropic

MODEL  = "claude-sonnet-4-20250514"
client = Anthropic()

print("\nAlles geladen ✓")

PROJECT_DIR  : /Users/habensebhatu/Downloads/School/Data_Science/Tweede_jaar/SEM_2/Datalab_4/vccr-travel-agent2/vccr_test
DB_PATH      : /Users/habensebhatu/Downloads/School/Data_Science/Tweede_jaar/SEM_2/Datalab_4/vccr-travel-agent2/vccr_test/data/agent.db
DATASET_PATH : /Users/habensebhatu/Downloads/School/Data_Science/Tweede_jaar/SEM_2/Datalab_4/vccr-travel-agent2/vccr_test/evaluatie/golden_dataset_f1.json

Alles geladen ✓


## 2. LLM — SQL genereren

In [52]:
print(SYSTEM_PROMPT[-600:])  # toont het einde van de prompt

a context-kolommen toe (geen aantal, status, datum, etc.) tenzij de gebruiker daar expliciet om vraagt.
6. Houd antwoorden kort en duidelijk
7. Bij vragen naar de "meeste", "duurste", "hoogste", "grootste" of "langste": gebruik altijd ORDER BY ... DESC LIMIT 1, zodat je precies één resultaat teruggeeft.
8. Bij vragen over de kaart (status, klasse, abonnement): gebruik LIMIT 1, tenzij de gebruiker naar alle kaarten vraagt.

## Antwoord formaat

Geef eerst de SQL query die je wilt uitvoeren in een ```sql``` code block.
Daarna geef ik je het resultaat, en dan formuleer je een duidelijk antwoord.



In [53]:
def genereer_sql(vraag: str, pers_nummer: str, groep: str) -> str | None:
    """Vraagt de LLM om een SQL-query en geeft alleen de SQL-string terug."""
    system = SYSTEM_PROMPT.format(pers_nummer=pers_nummer, groep=groep)
    response = client.messages.create(
        model=MODEL,
        max_tokens=1000,
        temperature=0,
        system=system,
        messages=[{"role": "user", "content": vraag}],
    )
    llm_antwoord = response.content[0].text
    if "```sql" in llm_antwoord:
        return llm_antwoord.split("```sql")[1].split("```")[0].strip()
    return None

## 3. Hulpfuncties

In [54]:
def voer_sql_uit(sql: str) -> list[tuple] | str:
    """Voert een SQL-query uit op de database. Geeft list[tuple] of een foutstring."""
    try:
        conn = sqlite3.connect(DB_PATH)
        cur  = conn.cursor()
        cur.execute(sql)
        rows = cur.fetchall()
        conn.close()
        return rows
    except Exception as e:
        return f"FOUT: {e}"


def normaliseer(rows: list[tuple]) -> list[tuple]:
    """Rond floats af op 2 decimalen en sorteert rijen (volgorde maakt niet uit)."""
    genormaliseerd = [
        tuple(round(v, 2) if isinstance(v, float) else v for v in rij)
        for rij in rows
    ]
    return sorted(genormaliseerd)


def vergelijk(goud: list[tuple], agent: list[tuple]) -> bool:
    return normaliseer(goud) == normaliseer(agent)

## 4. Evaluatieloop

In [55]:
def evalueer():
    with open(DATASET_PATH, encoding="utf-8") as f:
        dataset = json.load(f)

    vragen  = dataset["vragen"]
    totaal  = len(vragen)
    correct = 0
    per_categorie: dict[str, list[bool]] = {}

    print()
    print("=" * 65)
    print("  EVALUATIE FUNCTIE 1: TEXT-TO-SQL — EXECUTION ACCURACY")
    print("=" * 65)

    for entry in vragen:
        idx        = entry["id"]
        vraag      = entry["vraag"]
        pers_nr    = entry["pers_nummer"]
        groep      = entry["groep"]
        gouden_sql = entry["gouden_sql"]
        categorie  = entry["categorie"]

        print(f"\n[{idx}/{totaal}] {vraag}")
        print(f"        Categorie  : {categorie}")

        # Stap 1: genereer SQL via de LLM
        try:
            gegenereerde_sql = genereer_sql(vraag, pers_nr, groep)
        except Exception as e:
            print(f"        Status     : FOUT (LLM-aanroep mislukt: {e})")
            per_categorie.setdefault(categorie, []).append(False)
            continue

        if gegenereerde_sql is None:
            print("        Status     : FOUT (agent gaf geen SQL terug)")
            per_categorie.setdefault(categorie, []).append(False)
            continue

        weergave_sql = gegenereerde_sql[:110] + ("…" if len(gegenereerde_sql) > 110 else "")
        print(f"        SQL agent  : {weergave_sql}")

        # Stap 2: voer beide queries uit
        print('gouden_sql',gouden_sql)
        print('gegenereerde_sql',gegenereerde_sql)
        goud_resultaat  = voer_sql_uit(gouden_sql)
        agent_resultaat = voer_sql_uit(gegenereerde_sql)

        # Stap 3: foutafhandeling
        if isinstance(agent_resultaat, str):
            print(f"        Status     : FOUT (SQL-fout: {agent_resultaat})")
            per_categorie.setdefault(categorie, []).append(False)
            continue

        if isinstance(goud_resultaat, str):
            print(f"        Status     : FOUT (Gouden SQL-fout: {goud_resultaat})")
            per_categorie.setdefault(categorie, []).append(False)
            continue

        # Stap 4: vergelijk genormaliseerde resultaten
        is_correct = vergelijk(goud_resultaat, agent_resultaat)

        if is_correct:
            correct += 1
            print(f"        Status     : CORRECT  ✓")
            print(f"        Resultaat  : {goud_resultaat}")
        else:
            print(f"        Status     : FOUT  ✗")
            print(f"        Goud       : {normaliseer(goud_resultaat)}")
            print(f"        Agent      : {normaliseer(agent_resultaat)}")

        per_categorie.setdefault(categorie, []).append(is_correct)

    # ── Samenvatting ────────────────────────────────────────────────────────
    accuracy = (correct / totaal) * 100

    print()
    print("=" * 65)
    print("  SAMENVATTING")
    print("=" * 65)
    print(f"  Totaal vragen  : {totaal}")
    print(f"  Correct        : {correct}")
    print(f"  Fout           : {totaal - correct}")
    print(f"  Accuracy       : {accuracy:.1f}%")
    print()
    print("  Per categorie:")
    for cat, resultaten in per_categorie.items():
        n_correct = sum(resultaten)
        n_totaal  = len(resultaten)
        pct       = (n_correct / n_totaal) * 100
        print(f"    {cat:<16} {n_correct}/{n_totaal}  ({pct:.0f}%)")
    print("=" * 65)
    print()

## 5. Uitvoeren

In [56]:
evalueer()


  EVALUATIE FUNCTIE 1: TEXT-TO-SQL — EXECUTION ACCURACY

[1/20] Hoeveel heb ik in totaal uitgegeven en in welke reisklasse ben ik ingedeeld?
        Categorie  : join_klasse_kosten
        SQL agent  : SELECT ROUND(SUM("Prijs incl. BTW"), 2) as totaal_uitgegeven
FROM transacties 
WHERE "Pers.nummer" = '10040';
gouden_sql SELECT ROUND(SUM(t."Prijs incl. BTW"),2) as totaal, k.Klasse FROM transacties t JOIN kaarten k ON t."Pers.nummer" = k.Personeelsnummer WHERE t."Pers.nummer" = '10040' AND k.Status = 'Actief'
gegenereerde_sql SELECT ROUND(SUM("Prijs incl. BTW"), 2) as totaal_uitgegeven
FROM transacties 
WHERE "Pers.nummer" = '10040';
        Status     : FOUT  ✗
        Goud       : [(2047.65, 2.0)]
        Agent      : [(2047.65,)]

[2/20] Hoeveel heb ik uitgegeven in juli 2024?
        Categorie  : datum_maand
        SQL agent  : SELECT ROUND(SUM("Prijs incl. BTW"), 2) as totaal_uitgegeven
FROM transacties 
WHERE "Pers.nummer" = '659001' …
gouden_sql SELECT ROUND(SUM("Prijs incl. BT